# All Ohio Bridges

In [1]:
import requests
import pandas as pd

In [7]:
import requests
import pandas as pd

def get_all_bridges_from_api(api_url, fields_to_request):
    """
    Queries an ArcGIS REST API endpoint with pagination to retrieve all features.

    Args:
        api_url (str): The URL of the REST API's query endpoint.
        fields_to_request (list): A list of field names to retrieve.

    Returns:
        pandas.DataFrame: A DataFrame containing the records from the endpoint.
    """
    offset = 0
    all_features = []
    
    # Join the list of fields into a comma-separated string for the API call
    fields_str = ",".join(fields_to_request)
    
    print("Starting data retrieval...")

    while True:
        params = {
            'where': '1=1',
            'outFields': fields_str,       # Use the specific list of fields
            'returnGeometry': 'false',
            'resultOffset': offset,
            'f': 'json'
        }

        try:
            response = requests.get(api_url, params=params, timeout=60)
            response.raise_for_status() 
            data = response.json()

            # ADDED: Check for error messages within the JSON response
            if 'error' in data:
                print(f"API returned an error: {data['error']['message']}")
                print(f"Details: {data['error'].get('details', 'No details provided.')}")
                break

        except requests.exceptions.RequestException as e:
            print(f"An error occurred during the API request: {e}")
            break

        features_on_page = data.get('features', [])
        
        if not features_on_page:
            break

        all_features.extend(features_on_page)
        offset += len(features_on_page)
        
        print(f"Retrieved {len(all_features)} records so far...")

        if not data.get('exceededTransferLimit', False):
            break

    if not all_features:
        print("No features were retrieved.")
        return pd.DataFrame()

    attributes_list = [feature['attributes'] for feature in all_features]
    df = pd.DataFrame(attributes_list)
    
    return df

In [28]:
# --- SCRIPT STARTS HERE ---

# The URL for the query endpoint
query_url = "https://gis.dot.state.oh.us/arcgis/rest/services/TIMS/Assets/MapServer/5/query"

# **Customize this list with the fields you want to analyze**
# I've selected some common ones to get you started.
fields = [
    # --- Identifiers ---
    'SFN',               # NBI 008: The unique Structure File Number (primary ID)
    'STR_LOC_CARRIED',   # NBI 007: The name of the road or facility on the bridge
    'COUNTY_CD',         # NBI 003: The county where the bridge is located
    'DISTRICT',
    'CTL_BEGIN_NBR',
    'ORIG_PROJ_NBR',
    'LATITUDE_DD',       # (16.01): Latitude for mapping
    'LONGITUDE_DD',      # (17.01): Longitude for mapping

    # --- Route & Service ---
    'RTE_ON_BRG_CD',     # (208): The type of route on the bridge (e.g., State Highway)
    'FUNC_CLAS_CD',      # NBI 026: Functional class (e.g., Interstate, Arterial, Local)
    'TYPE_SERV1_CD',     # NBI 042A: Type of service ON the bridge (e.g., highway, railroad)
    'TYPE_SERV2_CD',     # NBI 042B: Type of service UNDER the bridge

    # --- Structure & Dimensions ---
    'MAIN_STR_MTL_CD',   # NBI 043A: Main structure material (e.g., Steel, Concrete, Wood)
    'MAIN_STR_TYPE_CD',  # NBI 043B: Main structure design type (e.g., Girder, Truss, Arch)
    'MAIN_SPANS',        # NBI 045: Number of spans in the main unit
    'APPRH_SPANS',       # NBI 046: Number of approach spans
    'MAX_SPAN_LEN',      # NBI 048: Length of the longest span
    'OVRL_STR_LEN',      # NBI 049: Overall length of the structure
    'BRG_RDW_WD',        # NBI 051: The curb-to-curb roadway width on the bridge
    'LANES_ON',          # NBI 028A: Number of lanes on the structure

    # --- Age & Traffic ---
    'YR_BUILT',          # (263): The year the bridge was originally built
    'MAJ_RECON_DT',      # (264): The year of the last major reconstruction
    'INVENT_RTE_ADT',    # NBI 029: Average Daily Traffic (how busy the bridge is)

    # --- Condition & Ratings ---
    'DECK_SUMMARY',      # NBI 058: Deck condition rating (scale of 0-9)
    'SUPS_SUMMARY',      # NBI 059: Superstructure condition rating
    'SUBS_SUMMARY',      # NBI 060: Substructure condition rating
    'GEN_APPRAISAL',     # (67.01): The lowest of the three summary ratings above
    'SUFF_RATING',       # Unofficial Sufficiency Rating (a 0-100 score of overall adequacy)
    'DEFIC_FUNC_RATING'  # Status: Whether the bridge is Structurally Deficient or Functionally Obsolete
]

# Fetch the data and create the DataFrame
bridges_df = get_all_bridges_from_api(query_url, fields)

# Display the results
if not bridges_df.empty:
    print("\n✅ Data retrieval complete!")
    print(f"Total number of bridges loaded: {len(bridges_df)}")
    
    print("\n--- DataFrame Head ---")
    print(bridges_df.head())

    print("\n--- DataFrame Info ---")
    bridges_df.info()

Starting data retrieval...
Retrieved 45334 records so far...

✅ Data retrieval complete!
Total number of bridges loaded: 45334

--- DataFrame Head ---
       SFN STR_LOC_CARRIED COUNTY_CD DISTRICT  CTL_BEGIN_NBR ORIG_PROJ_NBR  \
0  0100226           SR 32       ADA       09         11.867         70278   
1  0101214           SR 41       ADA       09         18.897                 
2  0101362           SR 41       ADA       09         20.918                 
3  0101710           US 52       ADA       09          5.438        053459   
4  0101842           US 52       ADA       09          6.794        033397   

   LATITUDE_DD  LONGITUDE_DD RTE_ON_BRG_CD FUNC_CLAS_CD  ... LANES_ON  \
0    38.935906    -83.469087            10           02  ...        2   
1    38.856397    -83.470597            10           06  ...        2   
2    38.883601    -83.456895            10           06  ...        2   
3    38.670917    -83.637339            10           02  ...        2   
4    38.684814 

In [29]:
bridges_df

,SFN,STR_LOC_CARRIED,COUNTY_CD,DISTRICT,CTL_BEGIN_NBR,ORIG_PROJ_NBR,LATITUDE_DD,LONGITUDE_DD,RTE_ON_BRG_CD,FUNC_CLAS_CD,...,LANES_ON,YR_BUILT,MAJ_RECON_DT,INVENT_RTE_ADT,DECK_SUMMARY,SUPS_SUMMARY,SUBS_SUMMARY,GEN_APPRAISAL,SUFF_RATING,DEFIC_FUNC_RATING
0,0100226,SR 32,ADA,09,11.867,70278,38.935906,-83.469087,10,02,...,2,268113600000,NaN,6641.0,6,7,7,7,094.5,0
1,0101214,SR 41,ADA,09,18.897,,38.856397,-83.470597,10,06,...,2,678340800000,NaN,3873.0,N,N,N,5,085.0,0
2,0101362,SR 41,ADA,09,20.918,,38.883601,-83.456895,10,06,...,2,1468209600000,NaN,3873.0,N,N,N,7,098.6,0
3,0101710,US 52,ADA,09,5.438,053459,38.670917,-83.637339,10,02,...,2,-299880000000,NaN,3140.0,N,N,N,6,098.1,0
4,0101842,US 52,ADA,09,6.794,033397,38.684814,-83.621439,10,02,...,2,899265600000,NaN,3294.0,8,6,8,6,098.8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45329,8847771,TH135b,WYA,01,4.692,,40.767569,-83.147814,42,09,...,2,1540872000000,NaN,120.0,8,8,8,8,094.4,0
45330,8848270,TH137a,WYA,01,0.391,,40.846358,-83.141092,42,09,...,2,1530417600000,NaN,155.0,8,8,8,8,092.5,0
45331,8848483,TH141a,WYA,01,0.519,,40.756647,-83.198528,42,09,...,1,-710193600000,NaN,209.0,N,N,N,6,099.9,0
45332,8849463,CH330,WYA,01,5.143,057148,40.840660,-83.146842,40,07,...,2,1120190400000,NaN,2051.0,7,7,7,7,099.4,0


In [30]:
# 01, 11: Interstate (Rural & Urban)
# 02, 12, 14: Other Principal Arterials / Freeways (Rural & Urban)
major_route_codes = ['01', '11', '02', '12', '14']

interstate_bridges_df = bridges_df[bridges_df['FUNC_CLAS_CD'].isin(major_route_codes)].copy()

# --- Display the results ---
print(f"Original number of bridges: {len(bridges_df)}")
print(f"Number of bridges on major routes: {len(interstate_bridges_df)}")

Original number of bridges: 45334
Number of bridges on major routes: 6920


In [31]:
# Condition 1: Bridge must be a single span
# This means it has 1 main span and 0 approach spans.
is_single_span = (interstate_bridges_df['MAIN_SPANS'] == 1) & \
                 (interstate_bridges_df['APPRH_SPANS'] == 0)

# Condition 2: The maximum span length is between 120' and 125'
has_correct_length = interstate_bridges_df['MAX_SPAN_LEN'].between(120, 125)

# Apply both conditions to the DataFrame
final_bridges_df = interstate_bridges_df[is_single_span & has_correct_length].copy()

In [32]:
# --- Display the results ---
print(f"Bridges found with a single span between 120' and 125': {len(final_bridges_df)}")

print("\n--- Results ---")
# Displaying key columns to verify both filters worked
print(final_bridges_df[[
    'SFN',
    'STR_LOC_CARRIED',
    'MAIN_SPANS',
    'APPRH_SPANS',
    'MAX_SPAN_LEN'
]])

Bridges found with a single span between 120' and 125': 20

--- Results ---
           SFN     STR_LOC_CARRIED  MAIN_SPANS  APPRH_SPANS  MAX_SPAN_LEN
440    0202037            IR 75 SB           1            0         121.0
5772   1700995               US 30           1            0         124.5
6175   1701002               US 30           1            0         124.5
8900   2511061  RAMP23 F2 OVER RAM           1            0         120.0
8946   2516535         SCHOOL WALK           1            0         120.0
9238   2134942       SAWMILL PKWY.           1            0         120.0
9497   2506149          I-670 Ramp           1            0         120.0
9768   2517922   I670SB TO AIRPORT           1            0         120.2
10156  2506068              CSX RR           1            0         124.0
10746  2517639            I-670 ML           1            0         123.1
17845  4808533             RAMP D4           1            0         124.0
21138  5708338         SB/NB IR 75  

In [33]:
import os
import tempfile
import subprocess
import sys

def edit_in_excel(df: pd.DataFrame) -> pd.DataFrame:
    """
    Saves a Pandas DataFrame to a temporary Excel file, opens it in the default
    application (like Excel), and reads the file back into a DataFrame after
    the user saves and closes it.

    This function will block execution until the external application (Excel) is closed.

    Args:
        df (pd.DataFrame): The DataFrame to be edited.

    Returns:
        pd.DataFrame: The edited DataFrame. Returns an empty DataFrame if the
                      original is empty or if an error occurs.
    
    Raises:
        Exception: If the default application for .xlsx files is not found or
                   if there's an issue opening the file.
    """
    if df.empty:
        print("Input DataFrame is empty. Returning an empty DataFrame.")
        return pd.DataFrame()

    # Use a temporary file that is not deleted on close
    # This allows us to pass the filename to an external application
    temp_file_descriptor, temp_file_path = tempfile.mkstemp(suffix=".xlsx")
    os.close(temp_file_descriptor) # Close the file handle

    try:
        # Save the DataFrame to the temporary Excel file, without the index
        df.to_excel(temp_file_path, index=False)

        print(f"Opening '{temp_file_path}' for editing. Please save and close the file when you are done.")

        # Open the file with the default application based on the OS
        if sys.platform == "win32":
            # For Windows
            os.startfile(temp_file_path)
        elif sys.platform == "darwin":
            # For macOS
            subprocess.run(["open", temp_file_path], check=True)
        else:
            # For Linux and other Unix-like OS
            subprocess.run(["xdg-open", temp_file_path], check=True)
        
        # Block until the user confirms they have saved and closed the file
        input("Press Enter after you have saved and closed the Excel file...")

        # Read the potentially modified data back from the Excel file
        print("Reading changes back into DataFrame...")
        edited_df = pd.read_excel(temp_file_path)

        return edited_df

    except FileNotFoundError:
        print("Error: Could not find a default application to open .xlsx files.")
        print("Please ensure you have a program like Excel or LibreOffice Calc installed.")
        return df # Return original dataframe
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return df # Return original dataframe
    finally:
        # Clean up by removing the temporary file
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)
            print(f"Temporary file '{temp_file_path}' has been deleted.")

In [34]:
if __name__ == '__main__':
    print("--- Original DataFrame ---")
    print("\n" + "="*30 + "\n")

    # 2. Call the function to open the DataFrame in Excel for editing
    # The script will pause here until you save and close Excel, then press Enter.
    edited_dataframe = edit_in_excel(final_bridges_df.copy()) # Pass a copy to preserve the original

    print("\n" + "="*30 + "\n")
    print("--- Edited DataFrame ---")
edited_dataframe

--- Original DataFrame ---


Opening 'C:\Users\dparks1\AppData\Local\Temp\tmpzbm4a5az.xlsx' for editing. Please save and close the file when you are done.


Press Enter after you have saved and closed the Excel file... 


Reading changes back into DataFrame...
Temporary file 'C:\Users\dparks1\AppData\Local\Temp\tmpzbm4a5az.xlsx' has been deleted.


--- Edited DataFrame ---


,SFN,STR_LOC_CARRIED,COUNTY_CD,DISTRICT,CTL_BEGIN_NBR,ORIG_PROJ_NBR,LATITUDE_DD,LONGITUDE_DD,RTE_ON_BRG_CD,FUNC_CLAS_CD,...,LANES_ON,YR_BUILT,MAJ_RECON_DT,INVENT_RTE_ADT,DECK_SUMMARY,SUPS_SUMMARY,SUBS_SUMMARY,GEN_APPRAISAL,SUFF_RATING,DEFIC_FUNC_RATING
0,202037,IR 75 SB,ALL,1,7.008,000213,40.730455,-84.075671,10,11,...,2,1387170000000,NaN,15822,9,9,9,9,96.5,0
1,1700995,US 30,CRA,3,1.375,019901,40.819116,-83.088920,10,2,...,2,1088654400000,NaN,4838,8,9,8,8,90.5,2
2,1701002,US 30,CRA,3,1.413,019901,40.819104,-83.088245,10,2,...,2,1088654400000,NaN,4838,8,9,8,8,90.5,2
3,2511061,RAMP23 F2 OVER RAM,FRA,6,0.129,NaN,40.112632,-83.018467,10,11,...,1,1512104400000,NaN,7100,9,9,7,7,96.3,0
4,2516535,SCHOOL WALK,FRA,6,12.740,052969,39.940836,-82.878364,99,14,...,0,15652800000,NaN,26084,8,8,8,8,NaN,2
5,2134942,SAWMILL PKWY.,DEL,6,2.770,NaN,40.176111,-83.094444,99,14,...,0,993960000000,NaN,14160,8,8,9,8,NaN,2
6,2506149,I-670 Ramp,FRA,6,0.128,005691,39.973062,-82.997064,10,14,...,2,747979200000,NaN,11345,6,8,8,8,94.9,2
7,2517922,I670SB TO AIRPORT,FRA,6,0.190,043608,40.000136,-82.920972,10,1,...,1,1258347600000,NaN,1730,8,8,8,8,97.6,0
8,2506068,CSX RR,FRA,6,1.960,019157,39.975455,-83.017847,99,12,...,0,-363038400000,NaN,69963,8,8,8,8,NaN,2
9,2517639,I-670 ML,FRA,6,3.291,003502,39.973259,-83.010223,10,11,...,8,1074747600000,NaN,99591,7,7,7,7,80.0,2


# Over Record Transition

[Python Object Oriented Programming Training](https://www.youtube.com/watch?v=ZDa-Z5JzLYM&list=PL-osiE80TeTsqhIuOqKhwlXsIBIdSeYtc) (88 mins)

147 records - from FHWA's [Data Crosswalk](https://www.fhwa.dot.gov/bridge/snbi/datacrosswalk.cfm)

To get access to the civilpy package, install python, then, in a terminal run `pip install civilpy`, after a lengthy first time installation, the below classes will be available.

In [ ]:
from civilpy.state.ohio.DOT.legacy import TimsBridge

In [ ]:
bridge = TimsBridge("6901573")

In [ ]:
from datetime import datetime
import pandas as pd
import math
import pint

In [ ]:
units = pint.UnitRegistry()

# 2701464, 6500609
sfn = 2701464

test_bridge = TimsBridge(sfn)
# test_bridge.__dict__

In [ ]:
fips_location = r"C:\Users\dparks1\Downloads\county_fips_master.csv"
fips_codes = pd.read_csv(fips_location, encoding="ISO-8859-1"
)

ohio_fips = pd.read_csv(
    "https://raw.githubusercontent.com/drparks71/civilpy/refs/heads/master/res/ohio_fips.csv"
)

In [ ]:
test_bridge_list = ["2701464", "6500609", "7704232"]

# //TODO - (1/30/2023 - reverify data source expanded) The first three SFN values don't work with historic data lookups in the NBIS
# Bad Values: '1739573', '7338597', '0101893', '0544353', '3402665'

# Configuration options (sets parameters to determine state preferences for certain decisions)
leading_zeros_user_input = 0  # Affects multiple conversions (BID01, BID03, ...)
state_user_input = "Ohio"

In [ ]:
first_bridge = TimsBridge(int(test_bridge_list[0]))

# District Records

In [ ]:
import os
import sys

In [ ]:
# Add the directory containing resources.py to the system path
script_dir = r"C:\Users\dparks1\PycharmProjects\SNBIUI"
if script_dir not in sys.path:
    sys.path.append(script_dir)

In [ ]:
in_filename = r"C:\Users\dparks1\PycharmProjects\SNBIUI\static\zephyr.svg"
out_filename = r"C:\Users\dparks1\PycharmProjects\SNBIUI\static\zephyr.ico"

In [ ]:
import cairosvg
from PIL import Image

def convert_svg_to_ico(svg_path, ico_path):
   """
   Convert an SVG file to an ICO file.

   :param svg_path: Path to the input SVG file.
   :param ico_path: Path to save the output ICO file.
   """
   # Step 1: Convert SVG to PNG in memory
   png_data = cairosvg.svg2png(url=svg_path)

   # Step 2: Save the PNG data and convert to multi-size ICO
   with open("temp_image.png", "wb") as f:
       f.write(png_data)

   # Step 3: Open the PNG with Pillow and save as ICO
   icon_sizes = [(16, 16), (32, 32), (48, 48), (64, 64), (128, 128), (256, 256)]  # Multi-resolution icon sizes
   with Image.open("temp_image.png") as image:
       image.save(ico_path, format="ICO", sizes=icon_sizes)

   print(f"ICO saved at: {ico_path}")

In [ ]:
# Example Usage
svg_file_path = filename  # Path to your input SVG file
ico_file_path = out_filename  # Path to save the output ICO file
convert_svg_to_ico(svg_file_path, ico_file_path)